In [3]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=5c05d2c8952646e00c9f65c18e8da87adcefacf7b8faf9bf025ba08f9c6eda00
  Stored in directory: /root/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
Successfully built seqeval


In [4]:
import ast
import json
import re
import time
import pandas as pd
import numpy as np
import torch
import os
import platform
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer,
    set_seed
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Устанавливаем seed для воспроизводимости
set_seed(42)

# Device settings

In [6]:
DEVICE = "cpu"
DEVICE_NAME = "CPU"
use_fp16 = False
if DEVICE == "cpu" and torch.cuda.is_available():
    DEVICE = "cuda"
    DEVICE_NAME = f"CUDA GPU - {torch.cuda.get_device_name()}"

MODEL_NAME = "DeepPavlov/rubert-base-cased"

# Annotations

In [7]:
LABEL_LIST = [
    "O",
    "B-TYPE", "I-TYPE",
    "B-BRAND", "I-BRAND", 
    "B-VOLUME", "I-VOLUME",
    "B-PERCENT", "I-PERCENT"
]

LABEL_TO_ID = {label: idx for idx, label in enumerate(LABEL_LIST)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}

In [8]:
volume_patterns = [
    r'\b(\d+(?:[.,]\d+)?\s*(?:л|лит|литр|литра|литров|литрах|литре))\b',
    r'\b(\d+(?:[.,]\d+)?\s*(?:мл|миллилитр|миллилитра|миллилитров))\b',
    r'\b(\d+(?:[.,]\d+)?\s*(?:г|гр|грам|грамм|грамма|граммов|граммах))\b',
    r'\b(\d+(?:[.,]\d+)?\s*(?:кг|килограмм|килограмма|килограммов))\b',
    r'\b(\d+(?:[.,]\d+)?\s*(?:шт|штук|штуки|штучек|штучки|штуку))\b',
    r'\b(\d+(?:[.,]\d+)?\s*(?:упак|упаковк|упаковка|упаковки|упаковок))\b',
    r'\b(\d+(?:[.,]\d+)?\s*(?:банк|банка|банки|банок|бутыл|бутылка|бутылки|бутылок|пакет|пакета|пакетов))\b'
]

percent_patterns = [
    r'\b(\d+(?:[.,]\d+)?\s*%)\b',
    r'\b(\d+(?:[.,]\d+)?\s*процент[а-я]*)\b',
    r'\b(\d+(?:[.,]\d+)?\s*жирност[а-я]*)\b'
]

VOLUME_RE = re.compile('|'.join(volume_patterns), re.I)
PERCENT_RE = re.compile('|'.join(percent_patterns), re.I)

In [9]:
def parse_annotations_safe(ann_str):
    """Безопасный парсинг аннотаций с проверками"""
    if not ann_str or ann_str.strip() in ("[]", "", "nan"):
        return []
    
    try:
        parsed = ast.literal_eval(ann_str)
        validated = []
        
        for item in parsed:
            if (isinstance(item, (list, tuple)) and 
                len(item) == 3 and 
                isinstance(item[0], int) and 
                isinstance(item[1], int) and 
                isinstance(item[2], str)):
                
                start, end, label = int(item[0]), int(item[1]), str(item[2])
                
                if label != 'O' and start < end:
                    validated.append((start, end, label))
        
        return validated
        
    except Exception as e:
        print(f"Ошибка парсинга аннотации: {ann_str} -> {e}")
        return []

def encode_example_improved(example, tokenizer):
    """Улучшенное кодирование с правильным выравниванием токенов"""
    text = example["sample"]
    annotations = parse_annotations_safe(example.get("annotation", ""))
    
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=128,
        padding='max_length'
    )
    
    offsets = encoding.pop("offset_mapping")
    labels = []
    
    for i, (token_start, token_end) in enumerate(offsets):
        if token_start == 0 and token_end == 0:
            labels.append(-100)
            continue
        
        assigned_label = "O"
        
        for start_char, end_char, entity_label in annotations:
            if not (token_end <= start_char or token_start >= end_char):
                if abs(token_start - start_char) <= 2:
                    if entity_label.startswith("B-"):
                        assigned_label = entity_label
                    else:
                        assigned_label = f"B-{entity_label}"
                else:
                    if entity_label.startswith("B-"):
                        assigned_label = f"I-{entity_label[2:]}"
                    elif entity_label.startswith("I-"):
                        assigned_label = entity_label  
                    else:
                        assigned_label = f"I-{entity_label}"
                break
        
        label_id = LABEL_TO_ID.get(assigned_label, LABEL_TO_ID["O"])
        labels.append(label_id)
    
    encoding["labels"] = labels
    return encoding

def compute_macro_f1(eval_pred):
    """Правильная метрика macro F1 как требует хакатон"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=-1)
    
    true_predictions = []
    true_labels = []
    
    for prediction, label in zip(predictions, labels):
        pred_seq = []
        true_seq = []
        
        for p, l in zip(prediction, label):
            if l != -100:
                pred_seq.append(ID_TO_LABEL[p])
                true_seq.append(ID_TO_LABEL[l])
        
        if pred_seq and true_seq:
            true_predictions.append(pred_seq)
            true_labels.append(true_seq)
    
    if not true_predictions:
        return {"macro_f1": 0.0, "overall_f1": 0.0}
    
    overall_f1 = f1_score(true_labels, true_predictions)
    overall_precision = precision_score(true_labels, true_predictions)  
    overall_recall = recall_score(true_labels, true_predictions)
    
    detailed_report = classification_report(true_labels, true_predictions, output_dict=True)
    
    entity_f1_scores = []
    entity_types = ["TYPE", "BRAND", "VOLUME", "PERCENT"]
    
    print("\nF1 по типам сущностей:")
    for entity_type in entity_types:
        if entity_type in detailed_report:
            f1_for_type = detailed_report[entity_type]["f1-score"]
            entity_f1_scores.append(f1_for_type)
            print(f"   {entity_type}: F1={f1_for_type:.3f}")
    
    macro_f1 = np.mean(entity_f1_scores) if entity_f1_scores else 0.0
    
    return {
        "macro_f1": macro_f1,
        "overall_f1": overall_f1,
        "precision": overall_precision,
        "recall": overall_recall,
        "num_entity_types": len(entity_f1_scores)
    }

# Data

In [10]:
csv_path = '/kaggle/input/x5-hack/train.csv'

df = pd.read_csv(csv_path, sep=';')

entity_stats = {"TYPE": 0, "BRAND": 0, "VOLUME": 0, "PERCENT": 0}
valid_samples = 0

for _, row in df.iterrows():
    annotations = parse_annotations_safe(row.get('annotation', ''))
    if annotations:
        valid_samples += 1
        for _, _, label in annotations:
            entity_type = label.replace('B-', '').replace('I-', '')
            if entity_type in entity_stats:
                entity_stats[entity_type] += 1

print(f"\n Статистика данных:")
print(f"   Образцов с аннотациями: {valid_samples}/{len(df)}")
for entity_type, count in entity_stats.items():
    if count > 0:
        print(f"   {entity_type}: {count} сущностей")

def get_entity_count(row):
    anns = parse_annotations_safe(row.get('annotation', ''))
    return min(len(anns), 3)

df['entity_count'] = df.apply(get_entity_count, axis=1)

train_df, val_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df['entity_count']
)

train_df = train_df.drop('entity_count', axis=1).reset_index(drop=True)
val_df = val_df.drop('entity_count', axis=1).reset_index(drop=True)
print(f"   Train: {len(train_df)}")
print(f"   Validation: {len(val_df)}")


 Статистика данных:
   Образцов с аннотациями: 26515/27251
   TYPE: 29060 сущностей
   BRAND: 7699 сущностей
   VOLUME: 84 сущностей
   PERCENT: 30 сущностей
   Train: 23163
   Validation: 4088


In [11]:
print(f"\n Токенизация {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_pandas(train_df).map(
    lambda ex: encode_example_improved(ex, tokenizer),
    batched=False,
    remove_columns=train_df.columns.tolist(),
    desc="Encoding train"
)

val_dataset = Dataset.from_pandas(val_df).map(
    lambda ex: encode_example_improved(ex, tokenizer),
    batched=False, 
    remove_columns=val_df.columns.tolist(),
    desc="Encoding validation"
)

print(f"   Train dataset: {len(train_dataset)}")
print(f"   Val dataset: {len(val_dataset)}")


 Токенизация DeepPavlov/rubert-base-cased


tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Encoding train:   0%|          | 0/23163 [00:00<?, ? examples/s]

Encoding validation:   0%|          | 0/4088 [00:00<?, ? examples/s]

   Train dataset: 23163
   Val dataset: 4088


# Train

In [12]:
import os
os.environ['WANDB_DISABLED'] = 'true' 
os.environ['WANDB_MODE'] = 'disabled'

def train_model(csv_path="train.csv"):
    """Обучение модели"""
    
    print("\nОБУЧЕНИЕ МОДЕЛИ")
    print(f"\nМодель: {MODEL_NAME}")
    
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABEL_LIST),
        ignore_mismatched_sizes=True
    )
    
    model = model.to(DEVICE)
    
    print(f"   Параметров: {sum(p.numel() for p in model.parameters()):,}")
    print(f"   Устройство модели: {next(model.parameters()).device}")
    

    if DEVICE == "cuda":
        train_batch_size = 8
        eval_batch_size = 16
        gradient_accum = 2
    else:
        train_batch_size = 4
        eval_batch_size = 8
        gradient_accum = 4
    
    training_args = TrainingArguments(
        output_dir="./intel_arc_ner_results",
        
        remove_unused_columns=False,
        
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps", 
        save_steps=100,
        logging_steps=50,
        
        num_train_epochs=4,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_steps=100,

        # fp16=True,
        
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,

        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        
        report_to=None,           # Отключаем все внешние логгеры
        logging_dir=None,         # Отключаем tensorboard
        run_name=None, 
    )
    
    # 7. Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_macro_f1,
        tokenizer=tokenizer,
    )
    
    print("\nНачало обучения...")
    print(f"   Устройство: {DEVICE_NAME}")
    print(f"   Batch size: {train_batch_size} (train), {eval_batch_size} (eval)")
    print("Available devices:", torch.cuda.device_count(), "GPU(s)")
    print("Current device:", torch.cuda.current_device(), torch.cuda.get_device_name(0))
    print("Model device:", next(model.parameters()).device)
    start_time = time.time()
    trainer.train()
    training_time = time.time() - start_time
    
    print(f"Обучение завершено за {training_time/60:.1f} минут")
    
    model_save_path = "./best_model"
    trainer.save_model(model_save_path)
    tokenizer.save_pretrained(model_save_path)
    
    print(f"\nМодель сохранена в: {model_save_path}")

    eval_results = trainer.evaluate()
    
    for key, value in eval_results.items():
        if isinstance(value, float):
            print(f"   {key}: {value:.4f}")
    
    final_macro_f1 = eval_results.get('eval_macro_f1', 0)
    print(f"\nИТОГОВЫЙ РЕЗУЛЬТАТ")
    print(f"Финальный Macro F1-score: {final_macro_f1:.4f}")
    print(f"Устройство: {DEVICE_NAME}")
    print(f"Время обучения: {training_time/60:.1f} минут")
    
    return trainer, eval_results, tokenizer

In [10]:
trainer, results, tokenizer = train_model("/kaggle/input/x5-hack/train.csv")


ОБУЧЕНИЕ МОДЕЛИ

Модель: DeepPavlov/rubert-base-cased


pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


   Параметров: 177,269,769
   Устройство модели: cuda:0

Начало обучения...
   Устройство: CUDA GPU - Tesla T4
   Batch size: 8 (train), 16 (eval)
Available devices: 2 GPU(s)
Current device: 0 Tesla T4
Model device: cuda:0


Step,Training Loss,Validation Loss,Macro F1,Overall F1,Precision,Recall,Num Entity Types
50,1.781900,1.246022,0.128479,0.391580,0.417671,0.368557,4
100,0.980500,0.658620,0.301614,0.667207,0.671452,0.663015,4
150,0.589400,0.444490,0.377191,0.814886,0.808046,0.821843,4
200,0.490900,0.395895,0.396111,0.841466,0.845503,0.837468,4
250,0.422200,0.355789,0.422016,0.879185,0.862299,0.896746,4
300,0.428100,0.331631,0.416231,0.878523,0.875780,0.881282,4
350,0.322900,0.313915,0.428519,0.890362,0.872930,0.908505,4
400,0.332500,0.316440,0.428954,0.885343,0.881386,0.889336,4
450,0.336300,0.291239,0.434151,0.897226,0.885611,0.909149,4
500,0.311900,0.280256,0.440536,0.905574,0.888046,0.923808,4



F1 по типам сущностей:
   TYPE: F1=0.453
   BRAND: F1=0.061
   VOLUME: F1=0.000
   PERCENT: F1=0.000

F1 по типам сущностей:
   TYPE: F1=0.724
   BRAND: F1=0.482
   VOLUME: F1=0.000
   PERCENT: F1=0.000

F1 по типам сущностей:
   TYPE: F1=0.867
   BRAND: F1=0.642
   VOLUME: F1=0.000
   PERCENT: F1=0.000

F1 по типам сущностей:
   TYPE: F1=0.884
   BRAND: F1=0.700
   VOLUME: F1=0.000
   PERCENT: F1=0.000

F1 по типам сущностей:
   TYPE: F1=0.914
   BRAND: F1=0.775
   VOLUME: F1=0.000
   PERCENT: F1=0.000

F1 по типам сущностей:
   TYPE: F1=0.918
   BRAND: F1=0.747
   VOLUME: F1=0.000
   PERCENT: F1=0.000

F1 по типам сущностей:
   TYPE: F1=0.926
   BRAND: F1=0.788
   VOLUME: F1=0.000
   PERCENT: F1=0.000

F1 по типам сущностей:
   TYPE: F1=0.917
   BRAND: F1=0.799
   VOLUME: F1=0.000
   PERCENT: F1=0.000

F1 по типам сущностей:
   TYPE: F1=0.927
   BRAND: F1=0.810
   VOLUME: F1=0.000
   PERCENT: F1=0.000

F1 по типам сущностей:
   TYPE: F1=0.933
   BRAND: F1=0.829
   VOLUME: F1=0.000
 


F1 по типам сущностей:
   TYPE: F1=0.958
   BRAND: F1=0.886
   VOLUME: F1=0.737
   PERCENT: F1=0.700
   eval_loss: 0.2521
   eval_macro_f1: 0.8201
   eval_overall_f1: 0.9400
   eval_precision: 0.9281
   eval_recall: 0.9522
   eval_runtime: 24.9820
   eval_samples_per_second: 163.6380
   eval_steps_per_second: 5.1240
   epoch: 4.0000

ИТОГОВЫЙ РЕЗУЛЬТАТ
Финальный Macro F1-score: 0.8201
Устройство: CUDA GPU - Tesla T4
Время обучения: 88.4 минут


In [13]:
from IPython.display import FileLink
import os
FileLink('bm.zip')

!zip -r bm.zip /kaggle/working/best_model
FileLink('bm.zip')

	zip warning: name not matched: /kaggle/working/best_model

zip error: Nothing to do! (try: zip -r bm.zip . -i /kaggle/working/best_model)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/kaggle/working/bm.zip

In [19]:
FileLink('bm.zip')

/kaggle/working/bm.zip

In [17]:
MODEL_PATH = "/kaggle/input/ner_model/pytorch/default/1/bm/kaggle/working/best_model"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH).to(DEVICE)

LABEL_LIST = ["O", "B-TYPE", "I-TYPE", "B-BRAND", "I-BRAND", "B-VOLUME", "I-VOLUME", "B-PERCENT", "I-PERCENT"]
ID_TO_LABEL = {i: label for i, label in enumerate(LABEL_LIST)}

volume_patterns = [r'\b(\d+(?:[.,]\d+)?\s*(?:л|лит|литр|литра|литров|мл|миллилитр|г|гр|грам|грамм|кг|килограмм|шт|штук|упак|упаковк|банк|банка|бутыл|бутылка|пакет))\b']
percent_patterns = [r'\b(\d+(?:[.,]\d+)?\s*%)\b', r'\b(\d+(?:[.,]\d+)?\s*процент[а-я]*)\b']

VOLUME_RE = re.compile('|'.join(volume_patterns), re.I)
PERCENT_RE = re.compile('|'.join(percent_patterns), re.I)

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128, return_offsets_mapping=True)
    offset_mapping = inputs.pop("offset_mapping")[0]
    inputs = {k: v.cuda() for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_classes = torch.argmax(outputs.logits, dim=-1)[0]
    
    entities = []
    current_entity = None
    
    for pred_class, offset in zip(predicted_classes, offset_mapping):
        if offset[0] == 0 and offset[1] == 0:
            continue
        label = ID_TO_LABEL[pred_class.item()]
        
        if label.startswith('B-'):
            if current_entity:
                entities.append(current_entity)
            current_entity = {'start': offset[0].item(), 'end': offset[1].item(), 'label': label[2:]}
        elif label.startswith('I-') and current_entity and current_entity['label'] == label[2:]:
            current_entity['end'] = offset[1].item()
        elif label != 'O' and current_entity and current_entity['label'] == label.replace('B-', '').replace('I-', ''):
            # ИСПРАВЛЕНИЕ: если тот же тип сущности продолжается
            current_entity['end'] = offset[1].item()
        else:
            if current_entity:
                entities.append(current_entity)
                current_entity = None
    
    if current_entity:
        entities.append(current_entity)
    
    # Объединяем соседние сущности одного типа
    merged_entities = []
    for entity in entities:
        if merged_entities and merged_entities[-1]['label'] == entity['label'] and merged_entities[-1]['end'] >= entity['start'] - 2:
            merged_entities[-1]['end'] = max(merged_entities[-1]['end'], entity['end'])
        else:
            merged_entities.append(entity)
    
    for match in VOLUME_RE.finditer(text):
        merged_entities.append({'start': match.start(), 'end': match.end(), 'label': 'VOLUME'})
    for match in PERCENT_RE.finditer(text):
        merged_entities.append({'start': match.start(), 'end': match.end(), 'label': 'PERCENT'})
    
    return str([(e['start'], e['end'], f"B-{e['label']}") for e in merged_entities])


df = pd.read_csv("/kaggle/input/x5-hack/submission.csv", sep=';')
df['annotation'] = [predict(row['sample']) for _, row in df.iterrows()]
df.to_csv("submission_predicted.csv", sep=';', index=False)